In [ ]:
# Install dependencies
!pip install sentence-transformers requests beautifulsoup4 pandas fredapi \
             scipy matplotlib statsmodels pandas-datareader -q

import warnings; warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time

FOMC_DATES = [
    # 2006
    "20060131","20060328","20060510","20060629","20060808","20060920","20061025","20061212",
    # 2007
    "20070131","20070321","20070509","20070618","20070807","20070918","20071031","20071211",
    # 2008
    "20080130","20080318","20080430","20080625","20080805","20080916","20081008","20081029","20081216",
    # 2009
    "20090128","20090318","20090429","20090624","20090812","20090923","20091104","20091216",
    # 2010
    "20100127","20100316","20100428","20100623","20100810","20100921","20101103","20101214",
    # 2011
    "20110126","20110315","20110427","20110622","20110809","20110921","20111102","20111213",
    # 2012
    "20120125","20120313","20120425","20120620","20120801","20120913","20121024","20121212",
    # 2013
    "20130130","20130320","20130501","20130619","20130731","20130918","20131030","20131218",
    # 2014
    "20140129","20140319","20140430","20140618","20140730","20140917","20141029","20141217",
    # 2015
    "20150128","20150318","20150429","20150617","20150729","20150917","20151028","20151216",
    # 2016
    "20160127","20160316","20160427","20160615","20160727","20160921","20161102","20161214",
    # 2017
    "20170201","20170315","20170503","20170614","20170726","20170920","20171101","20171213",
    # 2018
    "20180131","20180321","20180502","20180613","20180801","20180926","20181108","20181219",
    # 2019
    "20190130","20190320","20190501","20190619","20190731","20190918","20191030","20191211",
    # 2020
    "20200129","20200303","20200315","20200429","20200610","20200729","20200916","20201105","20201216",
    # 2021
    "20210127","20210317","20210428","20210616","20210728","20210922","20211103","20211215",
    # 2022
    "20220126","20220316","20220504","20220615","20220727","20220921","20221102","20221214",
    # 2023
    "20230201","20230322","20230503","20230614","20230726","20230920","20231101","20231213",
    # 2024
    "20240131","20240320","20240501","20240612","20240731","20240918","20241107","20241218",
    # 2025
    "20250129","20250319","20250507","20250618","20250730","20250917","20251029","20251210",
]

DATE_CORRECTIONS = {'2007-06-18': '2007-06-28'}

In [ ]:
#Block 2: Scraper & Corpus Builder
def scrape_statement(date_str):
    urls_to_try = [
        f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date_str}a.htm",
        f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date_str}default.htm",
        f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date_str}b.htm",
        f"https://www.federalreserve.gov/boarddocs/press/monetary/{date_str[:4]}/{date_str}/default.htm",
    ]
    for url in urls_to_try:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code != 200:
                continue
            soup = BeautifulSoup(r.content, 'html.parser')
            for tag in soup.find_all(['table', 'sup', 'nav', 'header', 'footer', 'script', 'style']):
                tag.decompose()
            content = (
                soup.find('div', {'id': 'article'}) or
                soup.find('div', {'class': 'col-xs-12 col-sm-8 col-md-8'}) or
                soup.find('div', {'class': re.compile(r'col-xs-12')}) or
                soup.find('div', {'id': 'content'}) or
                soup.find('td',  {'class': 'normal'}) or
                soup.find('body')
            )
            if not content:
                continue

            # Walk block-level tags explicitly to preserve paragraph breaks.
            # get_text(separator='\n') doesn't reliably produce newlines because
            # strip=True collapses inter-element whitespace before joining.
            BLOCK_TAGS = {
                'p', 'div', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6',
                'li', 'br', 'tr', 'blockquote', 'section', 'article'
            }
            chunks = []
            for elem in content.descendants:
                if getattr(elem, 'name', None) in BLOCK_TAGS:
                    chunk = elem.get_text(' ', strip=True)
                    if chunk:
                        chunks.append(chunk)

            # Fallback: if block-walking produced nothing, use raw get_text
            if not chunks:
                chunks = [content.get_text(' ', strip=True)]

            # Deduplicate chunks (block-walking visits nested elements,
            # so the same text can appear multiple times)
            seen = set()
            clean_chunks = []
            for chunk in chunks:
                if chunk not in seen:
                    seen.add(chunk)
                    clean_chunks.append(chunk)

            text = '\n\n'.join(clean_chunks)

            # Clean boilerplate
            for pattern in [
                r'for immediate release\s*',
                r'voting for this action.*',
                r'voting against.*',
                r'implementation note.*',
                r'last update.*',
            ]:
                text = re.sub(pattern, '', text, flags=re.IGNORECASE | re.DOTALL)

            # Normalise whitespace while preserving paragraph breaks
            text = re.sub(r'[^\S\n]+', ' ', text)       # spaces/tabs → single space
            text = re.sub(r'\n{3,}', '\n\n', text)       # 3+ newlines → double newline
            text = text.strip()

            if len(text) > 100:
                return text, url
        except Exception:
            continue
    return None, None


def build_corpus():
    print(f"Scraping {len(FOMC_DATES)} FOMC statements...\n")
    corpus, failed = [], []
    for i, date_str in enumerate(FOMC_DATES):
        date = pd.to_datetime(date_str, format='%Y%m%d')
        print(f"  [{i+1:>3}/{len(FOMC_DATES)}] {date.date()}", end=' ')
        text, url = scrape_statement(date_str)
        if text:
            corpus.append({'date': date, 'date_str': date_str, 'url': url,
                           'text': text, 'word_count': len(text.split())})
            print(f"✓  ({len(text.split())} words)")
        else:
            failed.append(date_str)
            print("✗  failed")
        time.sleep(0.4)

    df = pd.DataFrame(corpus).sort_values('date').reset_index(drop=True)
    for wrong, correct in DATE_CORRECTIONS.items():
        df.loc[df['date'] == pd.Timestamp(wrong), 'date'] = pd.Timestamp(correct)
    df = df.sort_values('date').reset_index(drop=True)
    df.to_csv('fomc_statements.csv', index=False)

    print(f"\n{'─'*50}")
    print(f"✅ Scraped:  {len(corpus)} | ❌ Failed: {len(failed)}")
    print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
    print(f"Avg words:  {df['word_count'].mean():.0f}")
    return df


corpus_df = build_corpus()

In [ ]:
#Block 3: Embeddings
from sentence_transformers import SentenceTransformer

print("Loading E5-Large model...")
model = SentenceTransformer('intfloat/e5-large-v2')

# Prefix required by E5 models for passage encoding
texts = ["passage: " + t for t in corpus_df['text'].tolist()]

print(f"Embedding {len(texts)} statements...")
embeddings = model.encode(texts, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
print(f"Embedding matrix shape: {embeddings.shape}")

np.save('fomc_embeddings.npy', embeddings)
print("Embeddings saved ✅")

In [ ]:
# Block 4: Build NVI
from scipy.spatial.distance import cosine as cosine_dist

def build_nvi(embeddings, df):
    """
    Compute Narrative Velocity Index (NVI) as cosine distance between
    consecutive statement embeddings. Z-scores use a true expanding window
    (no look-ahead bias): at time t, only data from rows 0..t are used.
    """
    nvi_values = [np.nan]
    for i in range(1, len(embeddings)):
        nvi_values.append(cosine_dist(embeddings[i-1], embeddings[i]))
    df['nvi'] = nvi_values

    # Expanding z-score — strictly no look-ahead
    nvi_zscore = [np.nan] * len(df)
    for t in range(len(df)):
        window = df['nvi'].iloc[:t+1].dropna()
        if len(window) >= 3:
            nvi_zscore[t] = (df['nvi'].iloc[t] - window.mean()) / window.std()
    df['nvi_zscore'] = nvi_zscore

    df['nvi_smooth'] = df['nvi'].rolling(3, min_periods=1).mean()
    return df


corpus_df = build_nvi(embeddings, corpus_df)
print("NVI built ✅")
print(corpus_df[['date', 'nvi', 'nvi_zscore']].dropna().head(8).to_string(index=False))

In [ ]:
# Block 5: Figure 1 — NVI Time Series
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150,
})

NAVY  = '#1B3A6B'; RED   = '#C0392B'
LGRAY = '#F4F6F9'; MGRAY = '#BDC3C7'; DGRAY = '#5D6D7E'

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor(LGRAY)

for start, end in [('2007-12-01', '2009-06-01'), ('2020-02-01', '2020-06-01')]:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color=MGRAY, alpha=0.4, zorder=0)

in_sample = corpus_df[corpus_df['date'] <= '2023-12-31']['nvi']
threshold  = in_sample.mean() + 1.5 * in_sample.std()
ax.axhline(y=threshold, color=RED, linestyle='--', linewidth=1.2,
           alpha=0.7, label='1.5σ threshold', zorder=1)
ax.fill_between(corpus_df['date'], corpus_df['nvi_smooth'], alpha=0.12, color=NAVY, zorder=2)
ax.plot(corpus_df['date'], corpus_df['nvi_smooth'],
        color=NAVY, linewidth=2, label='NVI (3-meeting smoothed)', zorder=3)

events = {
    '2008-10-01': ('GFC\nResponse',          0.95),
    '2013-06-01': ('Taper\nTantrum',          0.85),
    '2020-03-01': ('COVID\nEmergency',        0.95),
    '2021-11-01': ('"Transitory"\nAbandoned', 0.75),
    '2022-03-01': ('Rate\nLiftoff',           0.85),
}
ymax = corpus_df['nvi_smooth'].max()
for date_str, (label, ypos) in events.items():
    date = pd.Timestamp(date_str)
    ax.axvline(x=date, color=DGRAY, alpha=0.5, linewidth=1, linestyle=':', zorder=1)
    ax.text(date, ymax * ypos, label, fontsize=8.5, ha='center', color=DGRAY,
            fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=MGRAY, alpha=0.8))

ax.set_xlabel('Date', fontsize=12, color=DGRAY)
ax.set_ylabel('Cosine Distance (NVI)', fontsize=12, color=DGRAY)
ax.set_title('Federal Reserve Narrative Velocity Index\nFOMC Statements 2006–2023',
             fontsize=15, fontweight='bold', color=NAVY, pad=15)
ax.tick_params(colors=DGRAY)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

rec_patch = mpatches.Patch(color=MGRAY, alpha=0.6, label='NBER Recession')
ax.legend(handles=[ax.lines[1], ax.lines[0], rec_patch], loc='upper right',
          fontsize=10, framealpha=0.9)
fig.text(0.99, 0.01,
         'Methodology: E5-Large embeddings + cosine distance | Data: federalreserve.gov',
         ha='right', va='bottom', fontsize=7, color=MGRAY, style='italic')

plt.tight_layout()
plt.savefig('fig1_nvi_timeseries.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure 1 saved ✅")

In [ ]:
# Block 6: Macro Data & BSW Surprises
# fredapi for macro controls; BSW for high-frequency policy surprises
from fredapi import Fred
from scipy.stats import mstats

ANALYSIS_START = '2006-01-01'
ANALYSIS_END   = '2023-12-31'

fred = Fred(api_key='yourkey')  # free at fred.stlouisfed.org

def fred_fetch(series_id, start=ANALYSIS_START, end=ANALYSIS_END, retries=3):
    for attempt in range(retries):
        try:
            data = fred.get_series(series_id, observation_start=start, observation_end=end)
            time.sleep(1)
            return data
        except ValueError as e:
            if 'Too Many Requests' in str(e):
                wait = 10 * (attempt + 1)
                print(f"Rate limited — waiting {wait}s (retry {attempt+1}/{retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed to fetch {series_id} after {retries} retries")

ffr       = fred_fetch('DFF').to_frame('ffr')
cpi_raw   = fred_fetch('CPIAUCSL').to_frame('CPIAUCSL')
unemp_raw = fred_fetch('UNRATE').to_frame('UNRATE')

cpi_raw['inflation']     = cpi_raw['CPIAUCSL'].pct_change(12) * 100
cpi_raw['inflation_gap'] = cpi_raw['inflation'] - 2.0  # deviation from 2% target

# Bauer-Swanson (2023) high-frequency surprises
# Download from: https://www.aeaweb.org/articles?id=10.1257/aer.20230038
bs = pd.read_excel('monetary-policy-surprises-data.xlsx',
                   sheet_name='FOMC (update 2023)', parse_dates=['Date'])
bs = (bs[['Date', 'MPS', 'MPS_ORTH']]
      .rename(columns={'Date': 'date', 'MPS': 'surprise', 'MPS_ORTH': 'path_surprise'})
      .dropna(subset=['surprise']))
bs = bs[(bs['date'] >= ANALYSIS_START) & (bs['date'] <= ANALYSIS_END)].copy()
print(f"BSW meetings loaded: {len(bs)}  ({bs['date'].min().date()} → {bs['date'].max().date()})")

In [ ]:
#Block 7: Merge & Feature Engineering
def get_macro(meeting_date, series, col):
    try:
        return series.loc[:meeting_date][col].iloc[-1]
    except Exception:
        return np.nan

def get_rate_change(meeting_date, rate_df):
    try:
        rates = rate_df.loc[:meeting_date]['ffr']
        return round(rates.iloc[-1] - rates.iloc[-2], 4)
    except Exception:
        return np.nan


# Filter corpus to analysis window, then merge surprises
corpus_df = corpus_df[
    (corpus_df['date'] >= pd.Timestamp(ANALYSIS_START)) &
    (corpus_df['date'] <= pd.Timestamp(ANALYSIS_END))
].copy().reset_index(drop=True)

corpus_df.drop(columns=['surprise', 'path_surprise', 'surprise_abs',
                         'surprise_abs_w', 'hawkish_surprise', 'large_surprise'],
               errors='ignore', inplace=True)
corpus_df = corpus_df.merge(bs, on='date', how='left')

# ── Merge quality audit ─────────────────────────────────────────
matched     = corpus_df['surprise'].notna().sum()
unmatched   = corpus_df['surprise'].isna().sum()
print(f"\nBSW merge audit:")
print(f"  Exact matches:  {matched}/{len(corpus_df)}")
print(f"  Unmatched NaN:  {unmatched}/{len(corpus_df)}")

# For unmatched rows, check if a BSW date exists within 2 days
unmatched_df = corpus_df[corpus_df['surprise'].isna()][['date']].copy()
bs_dates     = pd.to_datetime(bs['date'].values)

near_misses = []
for _, row in unmatched_df.iterrows():
    diffs = abs((bs_dates - row['date']).days)
    closest_idx = diffs.argmin()
    closest_gap = diffs[closest_idx]
    if closest_gap <= 2:
        near_misses.append({
            'fomc_date':  row['date'].date(),
            'bsw_date':   bs_dates[closest_idx].date(),
            'gap_days':   closest_gap
        })

if near_misses:
    print(f"\n  ⚠️  Near-misses (BSW date within 2 days of FOMC date):")
    for nm in near_misses:
        print(f"     FOMC {nm['fomc_date']} ↔ BSW {nm['bsw_date']} "
              f"(gap: {nm['gap_days']} day{'s' if nm['gap_days'] > 1 else ''})")
    print(f"\n  Add these to DATE_CORRECTIONS in Block 1 to fix them.")
else:
    print(f"\n  ✅ No near-misses — unmatched dates have no BSW entry within 2 days")
    print(f"     (these are likely inter-meeting actions or pre-2006 dates)")

print()

# Derived surprise columns
corpus_df['surprise_abs'] = corpus_df['surprise'].abs()

non_null   = corpus_df['surprise_abs'].dropna()
winsorized = mstats.winsorize(non_null, limits=[0.05, 0.05])
corpus_df['surprise_abs_w'] = np.nan
corpus_df.loc[non_null.index, 'surprise_abs_w'] = winsorized

corpus_df['hawkish_surprise'] = (corpus_df['surprise'] > 0.05).astype(int)
# Large surprise = above expanding 75th percentile (no look-ahead)
corpus_df['large_surprise'] = (
    corpus_df['surprise_abs'] > corpus_df['surprise_abs'].expanding().quantile(0.75)
).astype(int)

# Macro controls (lagged one meeting to avoid look-ahead)
corpus_df['inflation_gap'] = corpus_df['date'].apply(lambda d: get_macro(d, cpi_raw, 'inflation_gap'))
corpus_df['unemployment']  = corpus_df['date'].apply(lambda d: get_macro(d, unemp_raw, 'UNRATE'))
corpus_df['rate_change']   = corpus_df['date'].apply(lambda d: get_rate_change(d, ffr))

corpus_df['nvi_lag1']     = corpus_df['nvi_zscore'].shift(1)
corpus_df['inf_gap_lag1'] = corpus_df['inflation_gap'].shift(1)
corpus_df['unemp_lag1']   = corpus_df['unemployment'].shift(1)

matched = corpus_df['surprise'].notna().sum()
print(f"BSW surprises matched: {matched}/{len(corpus_df)} meetings")
print("\n✅ Macro controls and surprises ready")

In [ ]:
# Block 8: Test 1 — NVI Predicts Rate Surprises
import statsmodels.api as sm
from scipy.stats import mannwhitneyu
from statsmodels.discrete.discrete_model import Probit

print("=" * 60)
print("TEST 1: DOES NVI PREDICT RATE SURPRISES?")
print("=" * 60)

clean = corpus_df.dropna(subset=['large_surprise', 'nvi_lag1', 'inf_gap_lag1', 'unemp_lag1'])

# A: Probit — does high lagged NVI predict large surprises?
X_p = sm.add_constant(clean[['nvi_lag1', 'inf_gap_lag1', 'unemp_lag1']])
probit = sm.Probit(clean['large_surprise'], X_p).fit(disp=False)
marginal_effects = probit.get_margeff()
ame_nvi = marginal_effects.margeff[0]
ame_p   = marginal_effects.pvalues[0]
print(f"\n── A: Probit AME (full sample) ──")
print(f"NVI AME = {ame_nvi:.4f}  p = {ame_p:.3f}  {'✅' if ame_p < 0.05 else '⚠️  not significant'}")

# B: Mann-Whitney — DESCRIPTIVE ONLY, not a hypothesis test
high_nvi = corpus_df[corpus_df['nvi_zscore'] >  1.0]['surprise_abs'].dropna()
low_nvi  = corpus_df[corpus_df['nvi_zscore'] <= 1.0]['surprise_abs'].dropna()

ratio_full = high_nvi.mean() / low_nvi.mean()
cohens_d   = ((high_nvi.mean() - low_nvi.mean()) /
               np.sqrt((high_nvi.std()**2 + low_nvi.std()**2) / 2))

print(f"\n── B: Descriptive — surprise size by NVI regime ──")
print(f"High NVI (n={len(high_nvi)}): mean = {high_nvi.mean():.4f}")
print(f"Low  NVI (n={len(low_nvi)}):  mean = {low_nvi.mean():.4f}")
print(f"Ratio = {ratio_full:.2f}×  |  Cohen's d = {cohens_d:.3f}")
print(f"(Mann-Whitney reported descriptively only — primary test is Probit above)")

# C: Robustness — exclude GFC, Probit only
print(f"\n── C: Robustness — excluding GFC (2007–2009) ──")
non_gfc  = corpus_df[~((corpus_df['date'] >= '2007-01-01') &
                        (corpus_df['date'] <= '2009-12-31'))].copy()
clean_ng = non_gfc.dropna(subset=['large_surprise', 'nvi_lag1',
                                   'inf_gap_lag1', 'unemp_lag1'])
high_nvi_ng = non_gfc[non_gfc['nvi_zscore'] >  1.0]['surprise_abs'].dropna()
low_nvi_ng  = non_gfc[non_gfc['nvi_zscore'] <= 1.0]['surprise_abs'].dropna()
ratio_ng    = high_nvi_ng.mean() / low_nvi_ng.mean()
cohens_ng   = ((high_nvi_ng.mean() - low_nvi_ng.mean()) /
                np.sqrt((high_nvi_ng.std()**2 + low_nvi_ng.std()**2) / 2))

if len(clean_ng) >= 20:
    X_ng      = sm.add_constant(clean_ng[['nvi_lag1', 'inf_gap_lag1', 'unemp_lag1']])
    probit_ng = sm.Probit(clean_ng['large_surprise'], X_ng).fit(disp=False)
    ame_ng    = probit_ng.get_margeff()
    ame_p_ng  = ame_ng.pvalues[0]
    ame_v_ng  = ame_ng.margeff[0]
    probit_ng_sig = ame_p_ng < 0.05
    print(f"Probit AME ex-GFC (n={len(clean_ng)}): "
          f"AME={ame_v_ng:.4f}  p={ame_p_ng:.3f}  "
          f"{'✅ survives' if probit_ng_sig else '⚠️  does not survive'}")
    print(f"Descriptive: ratio={ratio_ng:.2f}×  d={cohens_ng:.3f}")

# ── PRIMARY CONCLUSION: Question 1 ──────────────────────────────
probit_sig = ame_p < 0.05
print(f"""
{'═'*60}
TEST 1 CONCLUSION  (primary test: Probit AME)
{'═'*60}
Probit AME = {ame_nvi:.4f}   p = {ame_p:.3f}   {'✅ significant' if probit_sig else '❌ not significant'}

{'NVI predicts large surprises after controlling for macro conditions.'
  if probit_sig else
  'NVI does not survive macro controls. The raw descriptive difference '
  '(ratio={:.2f}x, d={:.2f}) reflects macro stress coinciding with '
  'high-NVI episodes rather than NVI independently predicting surprises.'.format(ratio_full, cohens_d)}

Descriptive (not tested): high-NVI meetings had {ratio_full:.2f}x
Larger raw surprises (Cohen d={cohens_d:.2f}) — driven by regime-change episodes, not a conditional signal.
{'═'*60}
""")

In [ ]:
#Block 9: Figure 2 — Test 1 Results
# Three panels: full-sample histogram, ex-GFC histogram, NVI timeline

TEAL = '#1A7F6E'
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

# Panel A: Full-sample distributions
ax = axes[0]
ax.set_facecolor(LGRAY); ax.spines[['top', 'right']].set_visible(False)
bins = np.linspace(0, 0.6, 25)
ax.hist(low_nvi,  bins=bins, alpha=0.65, color=NAVY, edgecolor='white', linewidth=0.5,
        label=f'Low NVI (n={len(low_nvi)})')
ax.hist(high_nvi, bins=bins, alpha=0.75, color=RED,  edgecolor='white', linewidth=0.5,
        label=f'High NVI (n={len(high_nvi)})')
ax.axvline(low_nvi.mean(),  color=NAVY, linestyle='--', linewidth=1.5,
           label=f'Low mean: {low_nvi.mean():.3f}')
ax.axvline(high_nvi.mean(), color=RED,  linestyle='--', linewidth=1.5,
           label=f'High mean: {high_nvi.mean():.3f}')
ax.text(0.97, 0.95,
        f'Full sample  (incl. GFC)\n─────────────────────\n'
        f'{ratio_full:.2f}× ratio  |  d={cohens_d:.2f}\n'
        f'MW: descriptive only\nProbit AME p={ame_p:.3f}  {"✅" if ame_p < 0.05 else "❌"}',
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5, multialignment='left',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=MGRAY))
ax.set_xlabel('Absolute Rate Surprise', fontsize=11, color=DGRAY)
ax.set_ylabel('Count', fontsize=11, color=DGRAY)
ax.set_title('Full Sample\n(incl. GFC 2007–09)', fontsize=12, fontweight='bold', color=NAVY)
ax.legend(fontsize=8.5, framealpha=0.9)

# Panel B: Ex-GFC distributions
ax2 = axes[1]
ax2.set_facecolor(LGRAY); ax2.spines[['top', 'right']].set_visible(False)
bins2 = np.linspace(0, 0.4, 22)
ax2.hist(low_nvi_ng,  bins=bins2, alpha=0.65, color=NAVY, edgecolor='white', linewidth=0.5,
         label=f'Low NVI (n={len(low_nvi_ng)})')
ax2.hist(high_nvi_ng, bins=bins2, alpha=0.75, color=TEAL, edgecolor='white', linewidth=0.5,
         label=f'High NVI ex-GFC (n={len(high_nvi_ng)})')
if not np.isnan(high_nvi_ng.mean()):
    ax2.axvline(low_nvi_ng.mean(),  color=NAVY, linestyle='--', linewidth=1.5)
    ax2.axvline(high_nvi_ng.mean(), color=TEAL, linestyle='--', linewidth=1.5)
    # NEW — robustness_holds and mw_p_ng gone, use probit ex-GFC instead
    ax2.text(0.97, 0.95,
             f'GFC-exclusion check\n─────────────────────\n'
             f'{ratio_ng:.2f}× ratio  |  d={cohens_ng:.2f}\n'
             f'MW: descriptive only\n'
             f'Probit ex-GFC p={ame_p_ng:.3f}  {"✅" if ame_p_ng < 0.05 else "❌"}',
             transform=ax2.transAxes, ha='right', va='top', fontsize=8.5, multialignment='left',
             bbox=dict(boxstyle='round', facecolor='white', edgecolor=MGRAY))
ax2.set_xlabel('Absolute Rate Surprise', fontsize=11, color=DGRAY)
ax2.set_ylabel('Count', fontsize=11, color=DGRAY)
ax2.set_title('Robustness: Excluding GFC\n(2007–09 removed)', fontsize=12, fontweight='bold', color=TEAL)
ax2.legend(fontsize=8.5, framealpha=0.9)

# Panel C: NVI timeline with large-surprise meetings flagged
ax3 = axes[2]
ax3.set_facecolor(LGRAY); ax3.spines[['top', 'right']].set_visible(False)
ax3.axvspan(pd.Timestamp('2007-01-01'), pd.Timestamp('2009-12-31'),
            color=MGRAY, alpha=0.3, label='GFC period')
ax3.plot(corpus_df['date'], corpus_df['nvi_zscore'],
         color=NAVY, linewidth=1, alpha=0.6, label='NVI Z-score')
ax3.axhline(y=1.0, color=RED, linestyle='--', linewidth=1, alpha=0.6,
            label='High NVI threshold (1σ)')
ax3.axhline(y=0,   color=DGRAY, linestyle='-', linewidth=0.5, alpha=0.4)
large = corpus_df[corpus_df['large_surprise'] == 1]
ax3.scatter(large['date'], large['nvi_zscore'],
            color=RED, s=40, zorder=5, alpha=0.8, label='Large surprise meeting')
ax3.set_xlabel('Date', fontsize=11, color=DGRAY)
ax3.set_ylabel('NVI Z-score', fontsize=11, color=DGRAY)
ax3.set_title('NVI Over Time\nwith Large Surprise Meetings Flagged',
              fontsize=12, fontweight='bold', color=NAVY)
ax3.legend(fontsize=8.5, framealpha=0.9)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax3.xaxis.set_major_locator(mdates.YearLocator(3))

fig.suptitle('Test 1: NVI Predicts Rate Surprises — Full Sample & GFC Robustness Check',
             fontsize=14, fontweight='bold', color=NAVY, y=1.02)
plt.tight_layout()
plt.savefig('fig2_test1_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure 2 saved ✅")

In [ ]:
#Block 10: Test 2 — Section-Level NVI
# Block 10: Section Splitter (rule-based: positional + regex)

import re
import os

# ── Anchor patterns from real FOMC statement language ──────────
SECTION_PATTERNS = {
    'decision': re.compile(
        r'(the (?:federal open market |)committee decided|'
        r'voted to|agrees to|'
        r'will maintain|will increase|will decrease|'
        r'will raise|will lower|will keep|'
        r'target range for the federal funds rate|'
        r'decided to (?:raise|lower|maintain|keep|increase|decrease))',
        re.IGNORECASE
    ),
    'forward_guidance': re.compile(
        r'(in determining|the committee expects|anticipates that|'
        r'will be appropriate|prepared to adjust|'
        r'ongoing increases will be|remain attentive|'
        r'in assessing the appropriate|'
        r'consistent with its statutory mandate|'
        r'committee will continue to monitor|'
        r'future policy(?:maker)?s? (?:will|may)|'
        r'pace of (?:future |)(?:increases|reductions|adjustments))',
        re.IGNORECASE
    ),
    'inflation': re.compile(
        r'(inflation|pce|price (?:stability|pressures|index|level)|'
        r'consumer price|core inflation|'
        r'longer.?run goal of 2 percent|'
        r'readings on inflation|'
        r'inflation has (?:risen|fallen|remained|moderated|eased)|'
        r'underlying inflation)',
        re.IGNORECASE
    ),
    'economic': re.compile(
        r'(information received|economic activity|labor market|'
        r'job gains|unemployment|'
        r'household spending|business (?:fixed )?investment|'
        r'gdp|gross domestic product|industrial production|'
        r'consumer spending|'
        r'pace of (?:economic )?(?:growth|expansion|recovery)|'
        r'real gdp|'
        r'economic conditions)',
        re.IGNORECASE
    ),
}

# Check decision/guidance first — their patterns are most distinctive.
# Inflation and economic patterns are broader so go last.
SECTION_PRIORITY = ['decision', 'forward_guidance', 'inflation', 'economic']


def split_by_position_and_regex(text, min_words=6):
    """
    The Fed's HTML doesn't paragraph-separate statement body text —
    the entire body arrives as one block. So we split on sentences
    and assign each to a section by regex.
    Positional fallback: first unmatched sentence → economic.
    Returns (dict of {section_name: text}, list of unmatched sentences).
    """
    header_noise = re.compile(
        r'^(?:january|february|march|april|may|june|july|august|'
        r'september|october|november|december|'
        r'for release|share\b|federal reserve issues|'
        r'fomc statement|for immediate release)',
        re.IGNORECASE
    )

    # Strip header lines before real content begins
    lines = text.split('\n')
    body_lines = []
    body_started = False
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if not body_started:
            if header_noise.match(line) or len(line.split()) < 6:
                continue
            body_started = True
        body_lines.append(line)
    body = ' '.join(body_lines)

    # Split body into sentences
    sentences = re.split(r'(?<=[.!?])\s+', body)
    sentences = [s.strip() for s in sentences if len(s.split()) >= min_words]

    labeled   = {sec: [] for sec in SECTION_PATTERNS}
    unmatched = []
    first_unmatched_used = False

    for sent in sentences:
        matched = False
        for sec in SECTION_PRIORITY:
            if SECTION_PATTERNS[sec].search(sent):
                labeled[sec].append(sent)
                matched = True
                break
        if not matched:
            if not first_unmatched_used:
                labeled['economic'].append(sent)
                first_unmatched_used = True
            else:
                unmatched.append(sent)

    return (
        {sec: ' '.join(sents) for sec, sents in labeled.items() if sents},
        unmatched
    )


def validate_splitter(corpus_df, n_sample=15, seed=42):
    sample = corpus_df.sample(min(n_sample, len(corpus_df)), random_state=seed)
    issues = 0

    for _, row in sample.iterrows():
        sections, unmatched = split_by_position_and_regex(row['text'])
        missing = [s for s in SECTION_PATTERNS if s not in sections]

        if missing:
            issues += 1
            print(f"  ⚠️  {row['date'].date()} — missing: {missing}")
            for s in missing:
                body = ' '.join(row['text'].split('\n'))
                sents = re.split(r'(?<=[.!?])\s+', body)
                hits = [se[:80] for se in sents if SECTION_PATTERNS[s].search(se)]
                if hits:
                    print(f"       Pattern hit but not captured: '{hits[0]}...'")
                else:
                    print(f"       No pattern match found at all for '{s}'")

    print(f"  {'✅' if issues == 0 else '⚠️ '} "
          f"{n_sample - issues}/{n_sample} statements fully labeled\n")
    return issues

# ── Run splitter (with cache) ───────────────────────────────────
if os.path.exists('fomc_sections.csv'):
    print("Loading cached sections...")
    sections_df = pd.read_csv('fomc_sections.csv')
    sections_df['date'] = pd.to_datetime(sections_df['date'])
else:
    print("Splitting statements into sections...")
    all_sections = []

    for i, row in corpus_df.iterrows():
        if i % 20 == 0:
            print(f"  {i+1}/{len(corpus_df)}: {row['date'].date()}")

        sections, _ = split_by_position_and_regex(row['text'])  # unpack tuple
        entry = {'date': row['date']}
        for sec in SECTION_PATTERNS:
            entry[f'text_{sec}']  = sections.get(sec, None)
            entry[f'words_{sec}'] = len((sections.get(sec) or '').split())
        all_sections.append(entry)

    sections_df = pd.DataFrame(all_sections)
    sections_df['date'] = pd.to_datetime(sections_df['date'])
    sections_df.to_csv('fomc_sections.csv', index=False)
    print("Saved ✅")

# ── Validate ────────────────────────────────────────────────────
print("\n=== Splitter validation ===")
n_issues = validate_splitter(corpus_df)

print("\n=== Section coverage ===")
for sec in SECTION_PATTERNS:
    nonempty  = sections_df[f'text_{sec}'].notna() & (sections_df[f'text_{sec}'] != '')
    avg_words = sections_df[f'words_{sec}'].mean()
    print(f"  {sec:>20}: {nonempty.sum()}/{len(sections_df)} statements  "
          f"({avg_words:.0f} avg words)")

In [ ]:
#Block 11: Section NVI Computation & Regressions
def true_expanding_zscore(series):
    """Z-score using only past + current observations — no look-ahead."""
    zscores = [np.nan] * len(series)
    vals = series.values
    for t in range(len(vals)):
        window = [v for v in vals[:t+1] if not np.isnan(v)]
        if len(window) >= 3:
            m, s = np.mean(window), np.std(window, ddof=1)
            if s > 0 and not np.isnan(vals[t]):
                zscores[t] = (vals[t] - m) / s
    return pd.Series(zscores, index=series.index)


print("Computing section-level NVIs...")
for sec in SECTION_PATTERNS:
    texts_sec = sections_df[f'text_{sec}'].fillna('').astype(str).tolist()
    texts_sec = [t if t.strip() and t != 'nan' else '' for t in texts_sec]
    sec_embs  = model.encode(
        ["passage: " + t if t else "passage: no content" for t in texts_sec],
        batch_size=32, show_progress_bar=False
    )
    dists = [np.nan]
    for i in range(1, len(sec_embs)):
        if texts_sec[i] and texts_sec[i-1]:
            dists.append(cosine_dist(sec_embs[i-1], sec_embs[i]))
        else:
            dists.append(np.nan)

    col = f'nvi_{sec}'
    sections_df[col]             = dists
    sections_df[f'{col}_z']      = true_expanding_zscore(sections_df[col])
    sections_df[f'{col}_z_lag1'] = sections_df[f'{col}_z'].shift(1)
    print(f"  {sec}: {sections_df[col].notna().sum()} distances computed")

# Merge surprise and macro data
sections_df['date'] = pd.to_datetime(sections_df['date'])
corpus_df['date']   = pd.to_datetime(corpus_df['date'])
test2_df = sections_df.merge(
    corpus_df[['date', 'surprise_abs', 'surprise_abs_w', 'inf_gap_lag1', 'unemp_lag1']],
    on='date', how='left'
)

# Regressions: full sample — ONE joint F-test replaces four separate tests
print("\n=== Question 2: Do sections jointly predict surprises? ===\n")

# Individual coefficients for plotting — stored but not individually tested
section_results = {}
for sec in SECTION_PATTERNS:
    col  = f'nvi_{sec}_z_lag1'
    data = test2_df.dropna(subset=['surprise_abs_w', col,
                                    'inf_gap_lag1', 'unemp_lag1'])
    if len(data) < 15:
        continue
    X   = sm.add_constant(data[[col, 'inf_gap_lag1', 'unemp_lag1']])
    fit = sm.OLS(data['surprise_abs_w'], X).fit(cov_type='HC3')
    ci  = fit.conf_int().loc[col]
    section_results[sec] = {
        'coef':  fit.params[col],
        'p_raw': fit.pvalues[col],   # descriptive only
        'ci_lo': ci[0],
        'ci_hi': ci[1],
        'r2':    fit.rsquared,
        'n':     len(data)
    }
    print(f"  {sec:>20}: coef={fit.params[col]:.4f}  "
          f"p={fit.pvalues[col]:.3f} (descriptive)  "
          f"R²={fit.rsquared:.3f}  n={len(data)}")

# ONE joint F-test — this is the single hypothesis test for Question 2
joint_cols = [f'nvi_{sec}_z_lag1' for sec in SECTION_PATTERNS]
joint_data = test2_df.dropna(
    subset=['surprise_abs_w'] + joint_cols + ['inf_gap_lag1', 'unemp_lag1']
)
X_joint  = sm.add_constant(joint_data[joint_cols + ['inf_gap_lag1', 'unemp_lag1']])
m_joint  = sm.OLS(joint_data['surprise_abs_w'], X_joint).fit(cov_type='HC3')
f_test   = m_joint.f_test([f'{c} = 0' for c in joint_cols])
f_pvalue = float(f_test.pvalue)
f_stat   = float(f_test.fvalue)
joint_sig = f_pvalue < 0.05

print(f"""
{'═'*60}
TEST 2 CONCLUSION  (primary test: joint F-test)
{'═'*60}
F-stat = {f_stat:.3f}   p = {f_pvalue:.3f}   n = {len(joint_data)}
{'✅ Sections jointly predict surprises' if joint_sig else '❌ Sections do not jointly predict surprises'}

Individual coefficients above are descriptive only.
{'═'*60}
""")
# ── Step 2: individual section follow-up ──────────────────────
# Only run if joint F-test is significant
# These are exploratory — BH corrected within this family only

if joint_sig:
    print("\n── Joint test significant → running section follow-up ──")
    print("   (BH correction within this family of 4 tests)\n")

    from statsmodels.stats.multitest import multipletests

    # Collect the 4 section p-values
    sec_labels = list(section_results.keys())
    sec_pvals  = [section_results[s]['p_raw'] for s in sec_labels]

    reject_sec, pvals_sec, _, _ = multipletests(
        sec_pvals, alpha=0.05, method='fdr_bh'
    )

    # Write back into section_results
    for sec, p_corr, sig in zip(sec_labels, pvals_sec, reject_sec):
        section_results[sec]['p_bh']    = p_corr
        section_results[sec]['sig_bh']  = bool(sig)
        marker = '✅' if sig else '❌'
        print(f"  {sec:>20}: coef={section_results[sec]['coef']:.4f}  "
              f"raw p={section_results[sec]['p_raw']:.3f}  "
              f"BH p={p_corr:.3f}  {marker}")

    print(f"\n  Sections surviving BH correction: "
          f"{sum(reject_sec)}/{len(reject_sec)}")

else:
    print("\n── Joint test not significant ──────────────────────────")
    print("   Individual section tests not run.")
    print("   No evidence that any section drives surprise size.")

    # Set sig_bh to False for all — figure needs this key
    for sec in section_results:
        section_results[sec]['p_bh']   = np.nan
        section_results[sec]['sig_bh'] = False

In [ ]:
#Block 12: Figure 3 — Test 2 Results
GREEN  = '#27AE60'; PURPLE = '#8E44AD'; RED_C = '#C0392B'
STEEL_BLUE = '#6C8EBF'; MID_GRAY = '#6C757D'; LIGHT_BG = '#F7F9FB'
COLORS = {'economic': NAVY, 'inflation': RED_C, 'decision': GREEN, 'forward_guidance': PURPLE}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Panel A: Section NVI over time
ax = axes[0]
ax.set_facecolor(LIGHT_BG); ax.spines[['top', 'right']].set_visible(False)
for sec, color in COLORS.items():
    s = sections_df[f'nvi_{sec}_z'].rolling(3, min_periods=1).mean()
    ax.plot(sections_df['date'], s, color=color, linewidth=1.6, alpha=0.9,
            label=sec.replace('_', ' ').title())
ax.axvline(pd.Timestamp('2022-01-01'), color=MID_GRAY, linestyle='--', linewidth=1, alpha=0.7)
yhi = max(ax.get_ylim()[1], 3.0)
ax.text(pd.Timestamp('2022-04-01'), yhi * 0.88,
        'Post-2022\nn too small\nfor inference',
        fontsize=7.5, color=MID_GRAY, ha='left', va='top',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor=MGRAY, alpha=0.8))
ax.axhline(0, color=MID_GRAY, linewidth=0.8, alpha=0.6)
ax.set_xlabel('Date', fontsize=11, color=DGRAY)
ax.set_ylabel('NVI Z-score (smoothed)', fontsize=11, color=DGRAY)
ax.set_title('Section-Level Narrative Velocity Over Time', fontsize=13, fontweight='bold', color=DGRAY)
ax.legend(frameon=False, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(3))
ax.grid(axis='y', linestyle='--', alpha=0.25); ax.set_axisbelow(True)

# Panel B: Section coefficients — one draw only
ax2 = axes[1]
ax2.set_facecolor(LIGHT_BG)
ax2.spines[['top', 'right']].set_visible(False)

sections_plot = [s for s in SECTION_PRIORITY if s in section_results]
coefs  = [section_results[s]['coef']  for s in sections_plot]
ci_lo  = [section_results[s]['ci_lo'] for s in sections_plot]
ci_hi  = [section_results[s]['ci_hi'] for s in sections_plot]
x      = np.arange(len(sections_plot))

if joint_sig:
    bar_colors = [NAVY if section_results[s]['sig_bh'] else MGRAY for s in sections_plot]
    title_str  = ('Section Coefficients — Follow-up exploration\n'
                  'BH-corrected within section family')
else:
    bar_colors = [MGRAY] * len(sections_plot)
    title_str  = ('Section Coefficients — Joint test not significant\n'
                  'Individual results not interpreted')

bars = ax2.bar(x, coefs, color=bar_colors, alpha=0.75,
               edgecolor='white', linewidth=0.5, width=0.55)

# Confidence intervals
for i, (lo, hi) in enumerate(zip(ci_lo, ci_hi)):
    ax2.plot([x[i], x[i]], [lo, hi], color=DGRAY, linewidth=1.8, zorder=5)
    ax2.plot([x[i]-0.08, x[i]+0.08], [lo, lo], color=DGRAY, linewidth=1.8, zorder=5)
    ax2.plot([x[i]-0.08, x[i]+0.08], [hi, hi], color=DGRAY, linewidth=1.8, zorder=5)

# n labels
for i, s in enumerate(sections_plot):
    ax2.text(x[i], min(ci_lo) - abs(min(ci_lo)) * 0.3,
             f"n={section_results[s]['n']}",
             ha='center', fontsize=8, color=DGRAY)

ax2.axhline(0, color=MID_GRAY, linewidth=0.8, alpha=0.6)
ax2.set_xticks(x)
ax2.set_xticklabels([s.replace('_', '\n').title() for s in sections_plot], fontsize=10)
ax2.set_ylabel('NVI Coefficient (OLS, HC3)', fontsize=11, color=DGRAY)
ax2.set_title(title_str, fontsize=12, fontweight='bold', color=DGRAY)

# Joint F-test annotation
f_color = NAVY if joint_sig else MID_GRAY
ax2.text(0.98, 0.97,
         f"Joint F-test\nF={f_stat:.2f}  p={f_pvalue:.3f}\n"
         f"{'✅ jointly significant' if joint_sig else '❌ not jointly significant'}",
         transform=ax2.transAxes, ha='right', va='top', fontsize=10,
         color=f_color, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=f_color, alpha=0.9))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=NAVY,  alpha=0.85, label='Significant after BH correction'),
    Patch(facecolor=MGRAY, alpha=0.85, label='Not significant after BH correction'),
]
ax2.legend(handles=legend_elements, fontsize=9, frameon=False, loc='upper left')
ax2.grid(axis='y', linestyle='--', alpha=0.25)
ax2.set_axisbelow(True)

fig.suptitle('Test 2: Section-Level NVI — Which Section Do Markets React To?',
             fontsize=14, fontweight='bold', color=NAVY, y=1.02)
for ax_i in axes:
    ax_i.tick_params(colors=DGRAY)
plt.tight_layout()
plt.savefig('fig3_test2_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure 3 saved ✅")

In [ ]:
#Block 13: Test 3 — NVI vs Word Count Horse Race
# Hawk-dove tone score as the word-count benchmark.
# Four-model OOS horse race + Diebold-Mariano test.
from scipy import stats as scipy_stats
from scipy.stats import mstats as _mstats
import pandas as pd

# Load Loughran-McDonald master dictionary
lm = pd.read_csv('Loughran-McDonald_MasterDictionary_1993-2025.csv')

# LM "Constraining" words map to hawkish Fed language
# LM "Negative" words map to dovish/risk language
# Convert to lowercase sets for fast lookup
lm_hawkish = set(
    lm[lm['Constraining'] != 0]['Word'].str.lower().tolist()
)
lm_dovish = set(
    lm[lm['Uncertainty'] != 0]['Word'].str.lower().tolist()
)

# Optionally augment with Fed-specific terms not in LM
# These are words the LM corpus (10-Ks) wouldn't capture
# because they're specific to central bank communication
FED_HAWKISH_SUPPLEMENT = {
    'tighten', 'tightening', 'restrictive', 'expeditious',
    'overshoot', 'overheating', 'liftoff'
}
FED_DOVISH_SUPPLEMENT = {
    'accommodative', 'transitory', 'stimulus',
    'patient', 'symmetric'
}

lm_hawkish = lm_hawkish | FED_HAWKISH_SUPPLEMENT
lm_dovish  = lm_dovish  | FED_DOVISH_SUPPLEMENT

def hawk_dove_score(text):
    """
    Hawk-dove tone using Loughran-McDonald finance dictionary
    supplemented with Fed-specific terms.
    Score = (hawkish - dovish) / (hawkish + dovish), range [-1, 1].
    """
    words = re.findall(r'\b[a-z]+\b', text.lower())
    h = sum(1 for w in words if w in lm_hawkish)
    d = sum(1 for w in words if w in lm_dovish)
    return (h - d) / (h + d) if (h + d) > 0 else 0.0

print(f"LM hawkish terms: {len(lm_hawkish)}")
print(f"LM dovish terms:  {len(lm_dovish)}")

corpus_df = corpus_df.sort_values('date').reset_index(drop=True)

corpus_df['hawk_dove']        = corpus_df['text'].apply(hawk_dove_score)
corpus_df['hawk_dove_z']      = true_expanding_zscore(corpus_df['hawk_dove'])
corpus_df['hawk_dove_z_lag1'] = corpus_df['hawk_dove_z'].shift(1)


# Train/test split: train on 2006-2018, test on 2019-2023
clean3 = corpus_df.dropna(subset=['surprise_abs', 'nvi_lag1', 'hawk_dove_z_lag1',
                                   'inf_gap_lag1', 'unemp_lag1'])
train = clean3[clean3['date'] <  '2019-01-01']
test  = clean3[clean3['date'] >= '2019-01-01']
print(f"Train: {len(train)} obs (2006–2018)  |  Test: {len(test)} obs (2019–2023)\n")
train = clean3[clean3['date'] < '2019-01-01'].copy()
test  = clean3[clean3['date'] >= '2019-01-01'].copy()

# winsorize using train only
w_lo = train['surprise_abs'].quantile(0.05)
w_hi = train['surprise_abs'].quantile(0.95)

train['surprise_abs_w'] = train['surprise_abs'].clip(w_lo, w_hi)
test['surprise_abs_w']  = test['surprise_abs'].clip(w_lo, w_hi)

# recreate clean3 for descriptive regressions
clean3['surprise_abs_w'] = clean3['surprise_abs'].clip(w_lo, w_hi)

model_specs = {
    'Word count only':  ['hawk_dove_z_lag1'],
    'NVI only':         ['nvi_lag1'],
    'NVI + Word count': ['nvi_lag1', 'hawk_dove_z_lag1'],
    'Full model':       ['nvi_lag1', 'hawk_dove_z_lag1', 'inf_gap_lag1', 'unemp_lag1'],
}

error_store   = {}
oos_results   = {}
benchmark_rmse = None

print(f"{'Model':<22} {'RMSE':>8} {'vs Baseline':>16}")
print("─" * 50)
for name, features in model_specs.items():
    m_fit  = sm.OLS(train['surprise_abs_w'], sm.add_constant(train[features])).fit()
    preds  = m_fit.predict(sm.add_constant(test[features]))
    errors = test['surprise_abs_w'].values - preds.values
    rmse   = np.sqrt(np.mean(errors**2))
    error_store[name] = errors
    oos_results[name] = {'rmse': rmse}
    if name == 'Word count only':
        benchmark_rmse = rmse
        vs = '— (baseline)'
    else:
        imp = (benchmark_rmse - rmse) / benchmark_rmse * 100
        vs  = f'{imp:+.1f}%  {"↓ better" if imp > 0 else "↑ WORSE"}'
    print(f"{name:<22} {rmse:>8.4f} {vs:>16}")

# NVI in combined model — reported descriptively, not tested
# (DM test below is the single hypothesis test for Question 3)
X_c  = sm.add_constant(clean3[['nvi_lag1', 'hawk_dove_z_lag1',
                                 'inf_gap_lag1', 'unemp_lag1']])
m_c  = sm.OLS(clean3['surprise_abs_w'], X_c).fit(cov_type='HC3')
corr = clean3[['nvi_lag1', 'hawk_dove_z_lag1']].corr().loc[
           'nvi_lag1', 'hawk_dove_z_lag1']

print(f"\n── Combined model (descriptive only) ──")
print(f"NVI coef        = {m_c.params['nvi_lag1']:.4f}  "
      f"p={m_c.pvalues['nvi_lag1']:.3f}  (not the primary test)")
print(f"Word count coef = {m_c.params['hawk_dove_z_lag1']:.4f}  "
      f"p={m_c.pvalues['hawk_dove_z_lag1']:.3f}")
print(f"Correlation NVI ↔ Word count: r={corr:.3f}")

# Diebold-Mariano test (single window)
def diebold_mariano(e1, e2):
    d  = e1**2 - e2**2
    dm = np.mean(d) / np.sqrt(np.var(d, ddof=1) / len(d))
    p  = 2 * (1 - scipy_stats.t.cdf(abs(dm), df=len(d)-1))
    return dm, p

dm_stat_val, dm_p = diebold_mariano(error_store['Word count only'], error_store['NVI only'])
print(f"""
{'═'*60}
TEST 3 CONCLUSION  (primary test: Diebold-Mariano OOS)
{'═'*60}
DM stat = {dm_stat_val:.3f}   p = {dm_p:.3f}
{'✅ NVI significantly better OOS' if dm_p < 0.05 else '❌ No significant OOS difference'}

NVI alone:    RMSE {oos_results['NVI only']['rmse']:.4f}
              ({(benchmark_rmse - oos_results['NVI only']['rmse'])/benchmark_rmse*100:+.1f}% vs word count)
Combined:     RMSE {oos_results['NVI + Word count']['rmse']:.4f}
              ({(benchmark_rmse - oos_results['NVI + Word count']['rmse'])/benchmark_rmse*100:+.1f}% vs word count)
Correlation NVI ↔ word count: r={corr:.3f} — complementary signals
{'═'*60}
""")

# Rolling Diebold-Mariano (24-meeting window, expanding train)
clean3_sorted = clean3.sort_values('date').reset_index(drop=True)
errors_wc_r, errors_nvi_r, dm_dates = [], [], []
start_train = 40
for t in range(start_train, len(clean3_sorted) - 1):
    tr, te = clean3_sorted.iloc[:t], clean3_sorted.iloc[t:t+1]
    m_wc  = sm.OLS(tr['surprise_abs_w'],
                   sm.add_constant(tr[['hawk_dove_z_lag1']], has_constant='add')).fit()
    m_nvi = sm.OLS(tr['surprise_abs_w'],
                   sm.add_constant(tr[['nvi_lag1']],         has_constant='add')).fit()
    actual      = te['surprise_abs_w'].values[0]
    errors_wc_r.append(actual - m_wc.predict( sm.add_constant(te[['hawk_dove_z_lag1']], has_constant='add')).values[0])
    errors_nvi_r.append(actual - m_nvi.predict(sm.add_constant(te[['nvi_lag1']],        has_constant='add')).values[0])
    dm_dates.append(te['date'].values[0])

window = 24
dm_stats_r, pvals_r, dates_r = [], [], []
for i in range(window, len(errors_wc_r)):
    e1, e2 = np.array(errors_wc_r[i-window:i]), np.array(errors_nvi_r[i-window:i])
    d  = e1**2 - e2**2
    dm = np.mean(d) / np.sqrt(np.var(d, ddof=1) / len(d))
    p  = 2 * (1 - scipy_stats.t.cdf(abs(dm), df=len(d)-1))
    dm_stats_r.append(dm); pvals_r.append(p); dates_r.append(dm_dates[i])

dm_df = pd.DataFrame({'date': dates_r, 'dm_stat': dm_stats_r, 'p': pvals_r})

In [ ]:
#Block 14: Figure 4 — Test 3 Results
SOFT_GRY = '#BFC7D5'; WARM_RED = '#C0392B'; ORANGE = '#E76F51'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

# Panel A: OOS RMSE bar chart
ax = axes[0]
ax.set_facecolor(LIGHT_BG); ax.spines[['top', 'right']].set_visible(False)
model_names = list(oos_results.keys())
rmses       = [oos_results[m]['rmse'] for m in model_names]
short_names = ['Word\nCount', 'NVI\nOnly', 'NVI +\nWord Count', 'Full\nModel']
bar_colors  = [SOFT_GRY, WARM_RED, TEAL, ORANGE]  # red = worse than baseline

bars = ax.bar(short_names, rmses, color=bar_colors, edgecolor='none', alpha=0.95)
rng  = max(rmses) - min(rmses)
for i, (bar, rmse) in enumerate(zip(bars, rmses)):
    # Improvement label above bar
    if i > 0:
        imp = (rmses[0] - rmse) / rmses[0] * 100
        lbl_color = WARM_RED if imp < 0 else TEAL
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + rng * 0.15,
                f'{imp:+.1f}%\n{"↑ worse" if imp < 0 else "↓ better"}',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold',
                color=lbl_color, linespacing=1.3)
    # RMSE value inside bar
    text_color = 'white' if bar_colors[i] != SOFT_GRY else DGRAY
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - rng * 0.25,
            f'{rmse:.4f}', ha='center', va='center',
            fontsize=9, fontweight='bold', color=text_color)

ax.set_ylim(min(rmses) - rng * 0.55, max(rmses) + rng * 0.6)
ax.set_ylabel('Out-of-Sample RMSE (zoomed)', fontsize=11, color=DGRAY)
ax.set_title('Horse Race: OOS Forecast Accuracy\nNVI alone is marginally worse; combined is best',
             fontsize=12, fontweight='bold', color=DGRAY)
ax.grid(axis='y', linestyle='--', alpha=0.25); ax.set_axisbelow(True)
ax.text(0.5, 0.97,
        f'NVI and word counts capture different\ndimensions of language change (r={corr:.2f})',
        transform=ax.transAxes, ha='center', va='top', fontsize=9, color=TEAL,
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=TEAL, alpha=0.85))

# Panel B: Rolling DM statistic
ax2 = axes[1]
ax2.set_facecolor(LIGHT_BG); ax2.spines[['top', 'right']].set_visible(False)
sig_mask = [p < 0.05 for p in dm_df['p']]
ax2.plot(dm_df['date'], dm_df['dm_stat'],
         color=NAVY, linewidth=1.8, alpha=0.9, label='DM statistic (+ = NVI better)')
ax2.fill_between(dm_df['date'], dm_df['dm_stat'], 0,
                 where=sig_mask, color=ORANGE, alpha=0.2, label='Significant at 5%')
ax2.axhline(0,  color=MID_GRAY, linewidth=0.8, alpha=0.6)
ax2.axhline( 2, color=MID_GRAY, linestyle=':', alpha=0.6, label='±2 critical value')
ax2.axhline(-2, color=MID_GRAY, linestyle=':', alpha=0.6)
ax2.set_xlabel('Date', fontsize=11, color=DGRAY)
ax2.set_ylabel('DM Statistic', fontsize=11, color=DGRAY)
ax2.set_title(f'Rolling Diebold-Mariano\n(NVI vs Word Count, {window}-meeting window)',
              fontsize=12, fontweight='bold', color=DGRAY)
ax2.legend(frameon=False, fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator(3))
ax2.grid(axis='y', linestyle='--', alpha=0.25); ax2.set_axisbelow(True)
wins = (dm_df['dm_stat'] > 0).mean()
ax2.text(0.02, 0.97,
         f'NVI numerically ahead {wins*100:.0f}% of windows\n'
         f'but significant {np.mean(sig_mask)*100:.0f}% of the time\n'
         f'→ difference is not statistically reliable',
         transform=ax2.transAxes, va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', edgecolor=MGRAY))

fig.suptitle('Test 3: NVI & Word Count Are Complementary Signals — Neither Dominates Alone',
             fontsize=13, fontweight='bold', color=NAVY, y=1.02)
for ax_i in axes:
    ax_i.tick_params(colors=DGRAY)
plt.tight_layout()
plt.savefig('fig4_test3_results.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("Figure 4 saved ✅")